# https://www.kaggle.com/datasets/odins0n/top-20-play-store-app-reviews-daily-update?select=Dropbox.csv

In [1]:
import polars as pl
import pandas as pd
from transformers import pipeline
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

F:\DataSpell\AI-with-jupyter\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_path = r"F:\Datasets\CSV datasets\Top 20 Play Store App Reviews (Daily Update)\Dropbox.csv"

In [3]:
df = pd.read_csv(df_path)

In [4]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

In [5]:
device = 0 if torch.cuda.is_available() else -1
nlp = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer, device=device)

Device set to use cuda:0


In [6]:
df_clean = df.dropna(subset=['content']).copy()
texts = df_clean['content'].astype(str).tolist()

In [7]:
try:
    results = nlp(texts, batch_size=32)
except Exception as e:
    print(f"Processing all at once failed: {e}")

    batch_size = 32
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_results = nlp(batch)
        results.extend(batch_results)

In [8]:
df_clean.loc[:, 'sentiment'] = [r['label'] for r in results]
df_clean.loc[:, 'confidence'] = [r['score'] for r in results]

In [12]:
df_clean[['content', 'sentiment', 'confidence']]

,content,sentiment,confidence
0,good,POSITIVE,0.999816
1,i love this app,POSITIVE,0.999861
2,My experience was wonderful,POSITIVE,0.999886
3,👍,NEGATIVE,0.697056
4,best .,POSITIVE,0.999825
...,...,...,...
9995,"Terrible, unable to download multiple file at ...",NEGATIVE,0.999736
9996,Useful app to have it on your mobile,POSITIVE,0.992190
9997,Drop and Go!! Even on The Go! Well you .....,POSITIVE,0.998687
9998,good job,POSITIVE,0.999836


In [10]:
df_clean

,reviewId,content,score,sentiment,confidence
0,bd1203a2-9e3d-4e81-b1b7-e349d0ef33ba,good,5,POSITIVE,0.999816
1,65e641df-0c87-4072-895e-f9228cd15642,i love this app,5,POSITIVE,0.999861
2,aea09a78-e16a-42c5-9d2e-c73bc819e8d5,My experience was wonderful,5,POSITIVE,0.999886
3,faa2f5f4-5b7a-4bf2-bdef-39bf8627d4c0,👍,5,NEGATIVE,0.697056
4,4c1df741-b784-4ade-b85c-551ef95b813e,best .,5,POSITIVE,0.999825
...,...,...,...,...,...
9995,38448b5a-f698-4f31-9b6c-7d1442c2475e,"Terrible, unable to download multiple file at ...",1,NEGATIVE,0.999736
9996,cb1ac42d-772e-423a-ac08-5893308bbe16,Useful app to have it on your mobile,5,POSITIVE,0.992190
9997,1cffe7d1-160d-4975-b8ca-35eefb82e656,Drop and Go!! Even on The Go! Well you .....,5,POSITIVE,0.998687
9998,a5af57ee-6b56-44d2-bd7e-7f6adaae8b48,good job,5,POSITIVE,0.999836


In [15]:
# df['sentiment'] = [r['label'] for r in results]
df_clean

,reviewId,content,score,sentiment,confidence
0,bd1203a2-9e3d-4e81-b1b7-e349d0ef33ba,good,5,POSITIVE,0.999816
1,65e641df-0c87-4072-895e-f9228cd15642,i love this app,5,POSITIVE,0.999861
2,aea09a78-e16a-42c5-9d2e-c73bc819e8d5,My experience was wonderful,5,POSITIVE,0.999886
3,faa2f5f4-5b7a-4bf2-bdef-39bf8627d4c0,👍,5,NEGATIVE,0.697056
4,4c1df741-b784-4ade-b85c-551ef95b813e,best .,5,POSITIVE,0.999825
...,...,...,...,...,...
9995,38448b5a-f698-4f31-9b6c-7d1442c2475e,"Terrible, unable to download multiple file at ...",1,NEGATIVE,0.999736
9996,cb1ac42d-772e-423a-ac08-5893308bbe16,Useful app to have it on your mobile,5,POSITIVE,0.992190
9997,1cffe7d1-160d-4975-b8ca-35eefb82e656,Drop and Go!! Even on The Go! Well you .....,5,POSITIVE,0.998687
9998,a5af57ee-6b56-44d2-bd7e-7f6adaae8b48,good job,5,POSITIVE,0.999836
